In [ ]:
# ── Colab Setup (run this first) ──────────────────────────────────────────
!pip install openai faiss-cpu langgraph langchain-core PyMuPDF tiktoken -q

import os
# Option 1: Paste your key directly
os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

Notebook 03 — Agent Pipeline (THE CORE)
AutoRFP-AI: Multi-Agent RFP Response System with LangGraph

WHAT THIS DOES:
- Wires Parser -> Draft -> Compliance agents via LangGraph
- Runs the full end-to-end pipeline on a sample RFP
- Shows retry/dedup logs (the "bug fix" demo)
- Outputs a formatted response document

COLAB SETUP:
import os; os.environ["OPENAI_API_KEY"] = "sk-..."

PREREQUISITES: Run Notebooks 01 and 02 first

### Setup — load everything from previous notebooks

In [ ]:
import os
import sys
import json
import time
import numpy as np
import faiss
from pathlib import Path
from openai import OpenAI

# Add src/ to import path
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
sys.path.insert(0, "src")
# If running from notebooks/ directory in Colab, also try parent
sys.path.insert(0, "..")

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

client = OpenAI()

# Load knowledge base
with open(DATA_DIR / "past_responses.json") as f:
    knowledge_base = json.load(f)
print(f"Knowledge base: {len(knowledge_base)} Q&A pairs")

# Load FAISS index
index = faiss.read_index(str(DATA_DIR / "faiss_index.bin"))
print(f"FAISS index: {index.ntotal} vectors")

# Load sample RFP
with open(DATA_DIR / "sample_rfp_text.txt") as f:
    sample_rfp_text = f.read()
print(f"Sample RFP: {len(sample_rfp_text)} chars")

### Rebuild retrieve() function

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"


def get_embeddings(texts):
    response = client.embeddings.create(input=texts, model=EMBEDDING_MODEL)
    return [item.embedding for item in response.data]


def retrieve(query, top_k=3, exclude_hashes=None):
    """Retrieve top-k similar past answers with dedup support."""
    if exclude_hashes is None:
        exclude_hashes = set()
    query_vec = np.array(get_embeddings([query]), dtype="float32")
    faiss.normalize_L2(query_vec)
    search_k = min(top_k + len(exclude_hashes) + 5, index.ntotal)
    scores, indices = index.search(query_vec, search_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        qa = knowledge_base[idx]
        if qa["hash"] in exclude_hashes:
            continue
        results.append({
            "id": qa["id"], "question": qa["question"],
            "answer": qa["answer"], "category": qa["category"],
            "score": float(score), "hash": qa["hash"],
        })
        if len(results) >= top_k:
            break
    return results


print("Retriever ready")

### Initialize agents

In [ ]:
from agents import ParserAgent, DraftAgent, ComplianceAgent

parser_agent = ParserAgent()
draft_agent = DraftAgent(retrieve_fn=retrieve)
compliance_agent = ComplianceAgent()

print("All agents initialized")

### Define LangGraph pipeline

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class RFPState(TypedDict):
    """State flowing through the pipeline."""
    rfp_text: str
    requirements: list[dict]
    drafts: list[dict]
    compliance_results: list[dict]
    final_output: list[dict]
    processing_log: list[str]


def parse_rfp(state: RFPState) -> dict:
    """Node 1: Parse RFP into structured requirements."""
    log = list(state.get("processing_log", []))
    log.append(">>> PARSER AGENT: Extracting requirements...")

    requirements = parser_agent.parse(state["rfp_text"])
    log.append(f"    Extracted {len(requirements)} requirements")
    return {"requirements": requirements, "processing_log": log}


def draft_responses(state: RFPState) -> dict:
    """Node 2: Draft answers for each requirement."""
    log = list(state.get("processing_log", []))
    drafts = []
    total = len(state["requirements"])
    log.append(f"\n>>> DRAFT AGENT: Drafting {total} responses...")

    for i, req in enumerate(state["requirements"]):
        log.append(f"\n    [{i+1}/{total}] {req['id']}: {req['question'][:60]}...")

        draft = draft_agent.draft_single(
            question=req["question"],
            requirement_id=req["id"],
        )
        drafts.append(draft)

        log.append(f"    Confidence={draft['confidence']:.2f} "
                   f"Attempts={draft['attempts']} "
                   f"Chunks={draft['context_chunks_used']}"
                   + (" ** FLAGGED **" if draft["flagged_for_review"] else ""))

    flagged = sum(1 for d in drafts if d["flagged_for_review"])
    log.append(f"\n    Drafted {len(drafts)} responses ({flagged} flagged)")
    return {"drafts": drafts, "processing_log": log}


def check_compliance(state: RFPState) -> dict:
    """Node 3: Validate all drafts against compliance rules."""
    log = list(state.get("processing_log", []))
    results = []
    log.append(f"\n>>> COMPLIANCE AGENT: Checking {len(state['drafts'])} drafts...")

    for draft in state["drafts"]:
        result = compliance_agent.check(draft)
        results.append(result)
        if not result["passed"]:
            flags = ", ".join(f["rule_id"] for f in result["flags"])
            log.append(f"    [{draft['requirement_id']}] FLAGGED: {flags}")
        else:
            log.append(f"    [{draft['requirement_id']}] Passed")

    passed = sum(1 for r in results if r["passed"])
    log.append(f"\n    {passed}/{len(results)} passed compliance")
    return {"compliance_results": results, "processing_log": log}


def assemble_output(state: RFPState) -> dict:
    """Node 4: Merge drafts + compliance into final output."""
    log = list(state.get("processing_log", []))
    log.append("\n>>> ASSEMBLY: Merging final output...")

    final = []
    for draft, comp in zip(state["drafts"], state["compliance_results"]):
        final.append({
            "requirement_id": draft["requirement_id"],
            "question": draft["question"],
            "answer": draft["answer"],
            "confidence": draft["confidence"],
            "attempts": draft["attempts"],
            "flagged_for_review": draft["flagged_for_review"],
            "compliance_passed": comp["passed"],
            "compliance_flags": comp.get("flags", []),
        })

    log.append(f"    {len(final)} responses assembled")
    return {"final_output": final, "processing_log": log}


# Build the graph
workflow = StateGraph(RFPState)
workflow.add_node("parse", parse_rfp)
workflow.add_node("draft", draft_responses)
workflow.add_node("compliance", check_compliance)
workflow.add_node("assemble", assemble_output)

workflow.add_edge(START, "parse")
workflow.add_edge("parse", "draft")
workflow.add_edge("draft", "compliance")
workflow.add_edge("compliance", "assemble")
workflow.add_edge("assemble", END)

app = workflow.compile()

print("LangGraph pipeline compiled")
print("Flow: START -> Parse -> Draft -> Compliance -> Assemble -> END")

### Run the full pipeline

In [ ]:
print("=" * 70)
print("RUNNING FULL AutoRFP PIPELINE")
print("=" * 70)

start_time = time.time()

result = app.invoke({
    "rfp_text": sample_rfp_text,
    "requirements": [],
    "drafts": [],
    "compliance_results": [],
    "final_output": [],
    "processing_log": [],
})

elapsed = time.time() - start_time

# Print processing log
print("\n-- Processing Log --")
for line in result["processing_log"]:
    print(line)

print(f"\nTotal time: {elapsed:.1f}s")

### Show retry/dedup log (THE BUG FIX DEMO)

In [ ]:
print("\n" + "=" * 70)
print("RETRY & DEDUPLICATION LOG")
print("(The 'bug fix' from the LinkedIn post)")
print("=" * 70)

if draft_agent.retry_log:
    for entry in draft_agent.retry_log:
        req_id = entry["requirement_id"]
        if entry["action"] == "DRAFT":
            print(f"  [{req_id}] Attempt {entry['attempt']} -> "
                  f"confidence={entry['confidence']:.2f}, "
                  f"chunks={entry['chunks_retrieved']}, "
                  f"total_accumulated={entry['total_chunks']}")
        elif entry["action"] == "DEDUP_STOP":
            print(f"  [{req_id}] DEDUP STOP at attempt {entry['attempt']} -> "
                  f"overlap={entry['overlap_ratio']:.0%}")
else:
    print("  No retries occurred")

total_attempts = sum(e.get("attempt", 0) for e in draft_agent.retry_log if e["action"] == "DRAFT")
total_q = len(result["final_output"])
dedup_stops = sum(1 for e in draft_agent.retry_log if e["action"] == "DEDUP_STOP")
print(f"\n  {total_attempts} drafting attempts for {total_q} questions")
print(f"  {dedup_stops} dedup stops")
print(f"  Avg attempts/question: {total_attempts/max(total_q,1):.1f}")

### Render final output document

In [ ]:
md = "# AutoRFP — Generated Response\n\n"
md += f"**Questions answered:** {len(result['final_output'])}  \n"
md += f"**Processing time:** {elapsed:.1f}s  \n\n---\n\n"

for entry in result["final_output"]:
    status = []
    if entry["flagged_for_review"]:
        status.append("NEEDS REVIEW")
    if not entry["compliance_passed"]:
        flags = ", ".join(f["rule_id"] for f in entry["compliance_flags"])
        status.append(f"Compliance: {flags}")
    status.append(f"Confidence: {entry['confidence']:.0%}")

    md += f"### {entry['requirement_id']}\n\n"
    md += f"**Q:** {entry['question']}\n\n"
    md += f"**A:** {entry['answer']}\n\n"
    md += f"*{' | '.join(status)} | Attempts: {entry['attempts']}*\n\n---\n\n"

# Display in Colab
try:
    from IPython.display import display, Markdown
    display(Markdown(md))
except ImportError:
    print(md)

# Save to file
output_path = DATA_DIR / "generated_rfp_response.md"
with open(output_path, "w") as f:
    f.write(md)
print(f"Saved to {output_path}")

### Pipeline statistics

In [ ]:
total = len(result["final_output"])
passed = sum(1 for e in result["final_output"] if e["compliance_passed"])
high_conf = sum(1 for e in result["final_output"] if e["confidence"] >= 0.7)
flagged = sum(1 for e in result["final_output"] if e["flagged_for_review"])
avg_conf = sum(e["confidence"] for e in result["final_output"]) / total

print("\n" + "=" * 70)
print("PIPELINE STATISTICS")
print("=" * 70)
print(f"  Requirements answered:   {total}")
print(f"  Compliance pass rate:    {passed}/{total} ({100*passed//total}%)")
print(f"  High-confidence (>=0.7): {high_conf}/{total} ({100*high_conf//total}%)")
print(f"  Flagged for review:      {flagged}/{total}")
print(f"  Average confidence:      {avg_conf:.2f}")
print(f"  Total time:              {elapsed:.1f}s ({elapsed/total:.1f}s per question)")
print(f"\nPipeline complete!")